In [3]:
from dotenv import load_dotenv

load_dotenv()

True

## Import libraries

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

C:\Users\Admin\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1a - Indexing (Document Ingestion)

In [5]:
video_id = "rbu7Zu5X1zI" # only the ID, not full URL
try:
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

The most common question I get about 3Blue1Brown is, what do I use to animate the videos? The short answer is that I wrote a custom Python library,  its name is Manim, so it's all programmatic and it's also very bespoke. What I wanted to do with this video is offer a behind the scenes to show you,  one, what Manim is for those who don't know, and even for those who do know,  to show a little bit about how I use it and what the workflow is. Now the way I'm doing this is I sat down with Ben Sparks when I was in the UK a couple  months ago, many of you may recognize him from his many great Numberphile videos. He wanted to know how Manim worked, I knew a number of other people had that same  question, so I recorded the conversation, it's kind of a scrappy recording but we'll  make do. And after a simple hello world example, we animate the famous Lorenz Attractor,  which is very important in the foundations of chaos,  it's also just a fun visual to get into. One quick thing I should mention

In [6]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='The most common question I get about 3Blue1Brown is, what do I use to animate the videos?', start=0.0, duration=4.62), FetchedTranscriptSnippet(text='The short answer is that I wrote a custom Python library, ', start=5.16, duration=3.194), FetchedTranscriptSnippet(text="its name is Manim, so it's all programmatic and it's also very bespoke.", start=8.354, duration=3.966), FetchedTranscriptSnippet(text='What I wanted to do with this video is offer a behind the scenes to show you, ', start=12.9, duration=3.935), FetchedTranscriptSnippet(text="one, what Manim is for those who don't know, and even for those who do know, ", start=16.835, duration=3.885), FetchedTranscriptSnippet(text='to show a little bit about how I use it and what the workflow is.', start=20.72, duration=3.28), FetchedTranscriptSnippet(text="Now the way I'm doing this is I sat down with Ben Sparks when I was in the UK a couple ", start=24.5, duration=4.205), Fetch

## Step 1b - Indexing (Text Splitting)

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [8]:
len(chunks)

74

In [9]:
chunks[50]

Document(metadata={}, page_content="And so now it's like as we're going, we can see it from all the different angles. I think the color scheme is still a little jarring to me. So I think what I might do is make it go to teal or something that's nice and close to it. And then maybe we'll make this actually even more aggressive. I'm not your best color audience. My color vision deficiency, definitely. I don't know if my colors are the same. Anyway, yellow and blue I can see apart. Blue and teal. In this case, like either you want them to be starkly different so  you see it or you want it to be aesthetically nice where maybe  you're just using the shade of it rather than the hue to distinguish. So I mean, one thing we could try, it's going to look cleaner,  but maybe be harder to discern. Let's say we go between two different shades of blue and I'm going  to now let's have some fun and let's have there be like 10 points. And so they're just going to change by their epsilon in the third co

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [10]:
embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en",
        encode_kwargs={"normalize_embeddings": True}
)

vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4681.77it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
vector_store.index_to_docstore_id

{0: '535ab58d-c44d-4a4c-a599-57540e6fffae',
 1: 'd9abb673-1422-4e25-bf88-695ae759fc1d',
 2: 'e8e71022-ee72-43bf-a415-04f82161229c',
 3: 'bd916d48-3b7c-453e-b38c-cdbac41f893d',
 4: 'cf7a9184-a830-4603-968d-cf65f5aaf0b4',
 5: '89b85e70-95f6-4862-b89e-a66f0a883140',
 6: 'e42a3e13-3dd3-43f4-a16f-9fd976a10f98',
 7: '841c4dc4-4f11-43ba-a002-3036e1375175',
 8: '8e46506a-ef3c-45c6-8c18-16bde143eb61',
 9: '9928addf-dcd6-49ce-8480-54896777e351',
 10: 'ae6f36be-e0c9-40f8-961f-2a41a3153193',
 11: 'f3646c68-d632-4d69-86a5-d0e4f57d945f',
 12: '7c369878-97f3-4834-9be9-26a8bfe85897',
 13: '287f32a0-a861-4cc7-8b0d-d874bf7d3115',
 14: '123f319d-dd3f-47f6-8e98-ce949af45a10',
 15: '8b16d884-3de9-416a-9c7f-f632e4fe5eb6',
 16: '486fceda-cfa3-4944-b833-1c52afbd7251',
 17: 'c35dc3c0-1dee-4e8d-a69c-2eed891263dd',
 18: '30f0c966-9ab8-4315-ba0f-d6fab455a7b1',
 19: 'b46796db-507e-4921-ab7a-01985cab0864',
 20: '44960c8b-10b6-4b73-a06f-3e0a03b56551',
 21: 'f7af4e50-8959-4d65-a46d-a4b20538b538',
 22: '16bffcd4-03b8-

In [45]:
vector_store.get_by_ids(['535ab58d-c44d-4a4c-a599-57540e6fffae'])

[Document(id='535ab58d-c44d-4a4c-a599-57540e6fffae', metadata={}, page_content="The most common question I get about 3Blue1Brown is, what do I use to animate the videos? The short answer is that I wrote a custom Python library,  its name is Manim, so it's all programmatic and it's also very bespoke. What I wanted to do with this video is offer a behind the scenes to show you,  one, what Manim is for those who don't know, and even for those who do know,  to show a little bit about how I use it and what the workflow is. Now the way I'm doing this is I sat down with Ben Sparks when I was in the UK a couple  months ago, many of you may recognize him from his many great Numberphile videos. He wanted to know how Manim worked, I knew a number of other people had that same  question, so I recorded the conversation, it's kind of a scrappy recording but we'll  make do. And after a simple hello world example, we animate the famous Lorenz Attractor,  which is very important in the foundations of c

## Step 2 - Retrieval

In [13]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [14]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000029687C37D50>, search_kwargs={'k': 4})

In [15]:
retriever.invoke('What is manim?')

[Document(id='bd916d48-3b7c-453e-b38c-cdbac41f893d', metadata={}, page_content="for others forked  the repo and created a version that is attentive to issues and pull requests and  has better testing and better documentation, and it's called the Manim community version. So it's generally recommended people start with that,  but in the meantime I often do make a bunch of changes or developments to my own,  over the last couple years I've made it more interactive, more performant,  things like that. The reason I bring this up is that what I'll be demoing with Ben is my own version,  but you should be aware that there is this divide and that if you want the better  documented and tested version, it's recommended to go with the community one for that. Without further ado, let's dive into the Hello World example. It's all in Python. Okay, and on the left is a Python, just a text file essentially with... Yep, so this is, it's Sublime text, I use Sublime, that's a text editor. We're going to 

## Step 3 - Augmentation

In [16]:
llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2
    )

In [17]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [31]:
question          = "is the topic of manim in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [32]:
retrieved_docs

[Document(id='fe6dd765-6b86-4cec-81dd-89bbdcd90de0', metadata={}, page_content="little bit more in here if you want where got this equation,  famous equation. If you wanted to say hey I just want to reference the E part of that. There's an animation called flash around and you'd be flashing around just at E. Or you want it. There's another one called indicate and you want to indicate that pie. Things like that. So you're pulling it out bit by bit. And in terms of understanding what functions like that exist. Obviously there's your head which is this vast  trove of experience over the time you wrote it. But the Manim community version is a different version but these still exist. Yeah Manim community definitely has much better documentation. There's not zero documentation on this one. You can see all the animations basically in a folder of the library called animation. If you just look through that folder you'll see the things that exist. All of the code I've ever written for any video 

In [33]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"little bit more in here if you want where got this equation,  famous equation. If you wanted to say hey I just want to reference the E part of that. There's an animation called flash around and you'd be flashing around just at E. Or you want it. There's another one called indicate and you want to indicate that pie. Things like that. So you're pulling it out bit by bit. And in terms of understanding what functions like that exist. Obviously there's your head which is this vast  trove of experience over the time you wrote it. But the Manim community version is a different version but these still exist. Yeah Manim community definitely has much better documentation. There's not zero documentation on this one. You can see all the animations basically in a folder of the library called animation. If you just look through that folder you'll see the things that exist. All of the code I've ever written for any video is on GitHub. If you go to github. slash 3b1b slash videos this GitHub\n\nfor o

In [34]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [35]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      little bit more in here if you want where got this equation,  famous equation. If you wanted to say hey I just want to reference the E part of that. There's an animation called flash around and you'd be flashing around just at E. Or you want it. There's another one called indicate and you want to indicate that pie. Things like that. So you're pulling it out bit by bit. And in terms of understanding what functions like that exist. Obviously there's your head which is this vast  trove of experience over the time you wrote it. But the Manim community version is a different version but these still exist. Yeah Manim community definitely has much better documentation. There's not zero documentation on this one. You can see all the animations basically in a folder of the library called animation. If you j

## Step 4 - Generation

In [36]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of Manim is discussed in this video. 

The discussion includes:
1. Introduction to Manim: It's a custom Python library used for animating videos.
2. Different versions of Manim: The creator's own version and the Manim community version, which has better documentation and testing.
3. Features and capabilities: Manim allows for programmatic animation and is very customizable.
4. Workflow: The creator demonstrates how to use Manim by showing a "Hello World" example and animating the Lorenz Attractor.
5. Rendering: The process of baking the animation into an MP4 file using a command-line interface.


## Building a Chain

In [37]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [38]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [39]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [40]:
parallel_chain.invoke('who is Demis')

{'context': "for others forked  the repo and created a version that is attentive to issues and pull requests and  has better testing and better documentation, and it's called the Manim community version. So it's generally recommended people start with that,  but in the meantime I often do make a bunch of changes or developments to my own,  over the last couple years I've made it more interactive, more performant,  things like that. The reason I bring this up is that what I'll be demoing with Ben is my own version,  but you should be aware that there is this divide and that if you want the better  documented and tested version, it's recommended to go with the community one for that. Without further ado, let's dive into the Hello World example. It's all in Python. Okay, and on the left is a Python, just a text file essentially with... Yep, so this is, it's Sublime text, I use Sublime, that's a text editor. We're going to be typing some Python here. I guess just as like a basic sense of w

In [41]:
parser = StrOutputParser()

In [42]:
main_chain = parallel_chain | prompt | llm | parser

In [43]:
main_chain.invoke('Can you summarize the video')

'The video appears to be about creating animations, possibly for educational videos. The creator is discussing their workflow, using a text editor, and sharing their code on GitHub. They are also experimenting with different animation effects, such as panning and fading, and discussing the use of specific functions and tools. The creator mentions that they will share more information, including the full conversation and code, on their Patreon page and GitHub repository.'